In [14]:
import warnings
from collections import Counter
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

print("라이브러리 로드 완료")

라이브러리 로드 완료


# 대표 장르 추출

## 목적
Steam 인디 게임 9,692개 중 리뷰 감성 분석을 위한 샘플링 대상 게임을 선별하기 위해, 각 게임의 대표 장르를 추출한다.

## 배경
- Steam `appdetails` API의 `genres` 필드는 알파벳 순으로 정렬된 멀티레이블 리스트로, "주 장르" 개념이 없음
- 층화 샘플링을 위해 게임당 하나의 대표 장르 배정이 필요
- 대표 장르 선정 방식: **희귀 장르 우선** — 게임이 가진 장르 중 전체 데이터에서 등장 빈도가 가장 낮은 장르를 대표로 선정

## 한계
- 희귀 장르가 반드시 해당 게임의 핵심 장르를 의미하지는 않음
- Sports/Racing 버킷에는 해당 장르가 부수적인 게임이 포함될 수 있음
- Steam 데이터 특성상 완전한 대표 장르 선정은 불가능하며, 이 한계를 감안하여 해석 필요

In [15]:
df = pd.read_csv('../../../data/processed/steam_indie_9692.csv')

## 1. 대상 장르 선정 및 대표 장르 추출

분석 대상 장르는 게임 수가 100개 이상이며 실제 게임 장르에 해당하는 8개로 한정한다.
(Indie는 거의 모든 게임에 포함되어 변별력이 없으므로 제외, Massively Multiplayer는 수가 적어 제외)

| 대상 장르 |
|----------|
| Action, Adventure, Casual, Simulation, RPG, Strategy, Sports, Racing |

In [16]:
import ast

TARGET_GENRES = {'Adventure', 'Casual', 'Action', 'Simulation', 'RPG', 'Strategy', 'Sports', 'Racing'}

# genres가 문자열로 저장된 경우 파싱
if isinstance(df['genres'].iloc[0], str):
    df['genres'] = df['genres'].apply(ast.literal_eval)

# 8개 대상 장르만 필터링
df['genres_filtered'] = df['genres'].apply(lambda g: [x for x in g if x in TARGET_GENRES])

# 대상 장르가 하나도 없는 게임 제외
df_filtered = df[df['genres_filtered'].map(len) > 0].copy()

# 8개 장르 내 희귀도 계산
genre_count = Counter(genre for genres in df_filtered['genres_filtered'] for genre in genres)
print('=== 장르별 게임 수 (희귀도 기준) ===')
for genre, count in sorted(genre_count.items(), key=lambda x: x[1]):
    print(f'  {genre:25s}: {count:,}')

# 희귀 장르 우선으로 대표 장르 선정
df_filtered['primary_genre'] = df_filtered['genres_filtered'].apply(
    lambda genres: min(genres, key=lambda g: genre_count[g])
)

print('\n=== 대표 장르 분포 ===')
print(df_filtered['primary_genre'].value_counts().to_string())

=== 장르별 게임 수 (희귀도 기준) ===
  Racing                   : 301
  Sports                   : 341
  Strategy                 : 2,005
  RPG                      : 2,194
  Simulation               : 2,503
  Action                   : 4,093
  Casual                   : 4,111
  Adventure                : 4,807

=== 대표 장르 분포 ===
primary_genre
Action        2282
Strategy      1912
RPG           1537
Simulation    1188
Casual        1091
Adventure      759
Racing         301
Sports         244


In [17]:
df_filtered[['appid', 'name', 'genres_filtered', 'primary_genre']].head(20)

,appid,name,genres_filtered,primary_genre
0,899770,Last Epoch,"[Action, Adventure, RPG]",RPG
1,251570,7 Days to Die,"[Action, Adventure, RPG, Simulation, Strategy]",Strategy
2,1116170,CyberCorp,"[Action, Adventure, RPG]",RPG
3,1326470,Sons Of The Forest,"[Action, Adventure, Simulation]",Simulation
4,2186680,"Warhammer 40,000: Rogue Trader","[Action, Adventure, RPG, Strategy]",Strategy
5,526870,Satisfactory,"[Adventure, Simulation, Strategy]",Strategy
6,2881650,Content Warning,"[Action, Adventure]",Action
7,513710,SCUM,"[Action, Adventure]",Action
8,1144200,Ready or Not,"[Action, Adventure]",Action
9,1145350,Hades II,"[Action, RPG]",RPG


## 2. 결과 확인

게임별 필터링된 장르 목록과 선정된 대표 장르를 확인한다.

In [18]:
genre_dist = df_filtered['primary_genre'].value_counts().reset_index()
genre_dist.columns = ['genre', 'count']

fig = px.bar(
    genre_dist,
    x='genre',
    y='count',
    text='count',
    title='대표 장르별 게임 수 분포',
    labels={'genre': '장르', 'count': '게임 수'},
)
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_categoryorder='total descending')
fig.show()